# 04 · Real-time ward alerts — which streets flood in the next 24 hours?

Pipeline each morning: pull the rainfall forecast → convert rain to runoff per catchment
(SCS Curve Number) → compare inflow volume against each sink's capacity → fill ratio →
green / amber / red per sink → aggregate to wards → map (+ optional Telegram push).

**Requires:** notebook 01 outputs. Rainfall comes from the free Open-Meteo API (no key);
you can swap in IMD nowcasts later — the interface is one function.

In [ ]:
import sys, subprocess, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install("rasterio", "osmnx", "geopandas", "shapely")

WORK = "/content/drive/MyDrive/floodtwin" if os.path.isdir("/content/drive/MyDrive") else "/content/floodtwin"
assert os.path.exists(f"{WORK}/sinks.csv"), "Run notebook 01 first."

AOI = (85.02, 25.54, 85.30, 25.68)
CELL_AREA = 900.0

## 1 · Rainfall forecast (next 24 h)
We sample five points (centre + corners) and take the max — conservative for alerts.

In [ ]:
import requests
import numpy as np

def forecast_rain_mm(lat, lon, hours=24):
    url = ("https://api.open-meteo.com/v1/forecast"
           f"?latitude={lat}&longitude={lon}&hourly=precipitation&forecast_days=2&timezone=Asia/Kolkata")
    js = requests.get(url, timeout=60).json()
    p = js["hourly"]["precipitation"][:hours]
    return float(sum(v or 0 for v in p))

pts = [((AOI[1] + AOI[3]) / 2, (AOI[0] + AOI[2]) / 2),
       (AOI[1], AOI[0]), (AOI[1], AOI[2]), (AOI[3], AOI[0]), (AOI[3], AOI[2])]
P = max(forecast_rain_mm(la, lo) for la, lo in pts)
print(f"Design rainfall for next 24 h (max over AOI): {P:.1f} mm")
# To test the system on a dry day, uncomment:
# P = 80.0   # simulate a heavy monsoon day

## 2 · Runoff per catchment — SCS Curve Number
CN depends on land cover (WorldCover) and hydrologic soil group (from clay %, notebook 02's
`clay.tif` if present, else group C assumed). Runoff depth
Q = (P − 0.2S)² / (P + 0.8S), with S = 25400/CN − 254 (mm).

In [ ]:
import rasterio

def read1(p):
    with rasterio.open(p) as s:
        return s.read(1), s.transform, s.height, s.width

wc, transform, R, C = read1(f"{WORK}/worldcover.tif")
labels, _, _, _ = read1(f"{WORK}/catchment_labels.tif")
r, c = min(wc.shape[0], labels.shape[0]), min(wc.shape[1], labels.shape[1])
wc, labels = wc[:r, :c], labels[:r, :c]

if os.path.exists(f"{WORK}/clay.tif"):
    clay = read1(f"{WORK}/clay.tif")[0][:r, :c] / 10.0
    sg = np.full(clay.shape, 2)                     # B
    sg[clay > 25] = 3                                # C
    sg[clay > 40] = 4                                # D
    sg[clay < 10] = 1                                # A
else:
    sg = np.full(wc.shape, 3)                        # assume group C (Gangetic silty clay)

CN_TABLE = {  # worldcover class -> CN for soil groups (A, B, C, D)
    10: (36, 60, 73, 79), 20: (45, 66, 77, 83), 30: (49, 69, 79, 84),
    40: (67, 78, 85, 89), 50: (89, 92, 94, 95), 60: (77, 86, 91, 94),
    80: (98, 98, 98, 98), 90: (98, 98, 98, 98),
}
cn = np.full(wc.shape, 80.0)
for cls, vals in CN_TABLE.items():
    for g in range(1, 5):
        cn[(wc == cls) & (sg == g)] = vals[g - 1]

S = 25400.0 / cn - 254.0
Ia = 0.2 * S
Q = np.where(P > Ia, (P - Ia) ** 2 / (P + 0.8 * S), 0.0)   # runoff depth, mm
print(f"mean runoff depth over city: {Q.mean():.1f} mm of {P:.1f} mm rain")

## 3 · Fill ratio per sink → alert level

In [ ]:
import pandas as pd

sinks = pd.read_csv(f"{WORK}/sinks.csv")
alerts = []
for _, s in sinks.iterrows():
    cat = labels == int(s.sink_id)
    if cat.sum() == 0:
        continue
    inflow_m3 = float(Q[cat].mean() / 1000.0 * cat.sum() * CELL_AREA)
    ratio = inflow_m3 / max(s.volume_m3, 1.0)
    level = "RED" if ratio >= 1.0 else ("AMBER" if ratio >= 0.5 else "GREEN")
    alerts.append(dict(sink_id=int(s.sink_id), lat=s.lat, lon=s.lon,
                       inflow_m3=round(inflow_m3), capacity_m3=s.volume_m3,
                       fill_ratio=round(ratio, 2), level=level))
al = pd.DataFrame(alerts).sort_values("fill_ratio", ascending=False)
al.to_csv(f"{WORK}/alerts_today.csv", index=False)
print(al.head(15).to_string())

## 4 · Aggregate to wards
Tries OSM administrative boundaries (admin_level 9/10) inside Patna; if OSM coverage is thin,
falls back to a 1 km grid so the product always renders. Swap in the official Patna Municipal
Corporation ward shapefile when you obtain it — one `read_file` line.

In [ ]:
import geopandas as gpd
from shapely.geometry import box, Point

bbox_poly = box(AOI[0], AOI[1], AOI[2], AOI[3])
wards = None
try:
    import osmnx as ox
    g = ox.features_from_polygon(bbox_poly, tags={"boundary": "administrative", "admin_level": ["9", "10"]})
    g = g[g.geometry.type.isin(["Polygon", "MultiPolygon"])].reset_index()
    if len(g) >= 5:
        wards = g[["geometry"]].copy()
        wards["ward"] = [f"ward_{i}" for i in range(len(wards))]
        print(f"OSM wards found: {len(wards)}")
except Exception as e:
    print("OSM ward fetch failed:", e)

if wards is None:
    cellsz = 0.01   # ~1.1 km
    cells = [box(x, y, x + cellsz, y + cellsz)
             for x in np.arange(AOI[0], AOI[2], cellsz)
             for y in np.arange(AOI[1], AOI[3], cellsz)]
    wards = gpd.GeoDataFrame({"ward": [f"zone_{i}" for i in range(len(cells))]},
                             geometry=cells, crs="EPSG:4326")
    print("using 1 km grid fallback:", len(wards), "zones")

pts = gpd.GeoDataFrame(al, geometry=[Point(lo, la) for la, lo in zip(al.lat, al.lon)], crs="EPSG:4326")
joined = gpd.sjoin(pts, wards.set_crs("EPSG:4326", allow_override=True), predicate="within")
score = {"GREEN": 0, "AMBER": 1, "RED": 2}
ward_level = joined.groupby("ward")["level"].agg(lambda s: max(s, key=lambda v: score[v]))
wards["alert"] = wards["ward"].map(ward_level).fillna("GREEN")

In [ ]:
import folium

colors = {"GREEN": "#2e7d32", "AMBER": "#ef6c00", "RED": "#c62828"}
m = folium.Map(location=[(AOI[1] + AOI[3]) / 2, (AOI[0] + AOI[2]) / 2],
               zoom_start=12, tiles="cartodbpositron")
for _, w in wards.iterrows():
    if w.alert == "GREEN":
        continue
    folium.GeoJson(w.geometry.__geo_interface__,
                   style_function=lambda f, col=colors[w.alert]:
                   dict(fillColor=col, color=col, weight=1, fillOpacity=0.35)).add_to(m)
for _, a in al.iterrows():
    folium.CircleMarker([a.lat, a.lon], radius=6, color=colors[a.level], fill=True,
        popup=f"sink {a.sink_id}: {a.level}<br>fill ratio {a.fill_ratio}").add_to(m)
m.save(f"{WORK}/alert_map.html")
m

## 5 · Optional: push the summary to Telegram
Create a bot with @BotFather, get your chat id from @userinfobot, fill the two constants.
Then run this notebook every morning (manually, or with Colab Pro scheduled notebooks, or by
moving the code to a tiny GitHub Actions cron — it has no GPU needs).

In [ ]:
TELEGRAM_TOKEN = ""   # <-- optional
TELEGRAM_CHAT = ""    # <-- optional
if TELEGRAM_TOKEN and TELEGRAM_CHAT:
    reds = al[al.level == "RED"]; ambers = al[al.level == "AMBER"]
    msg = (f"Patna flood outlook, next 24h: {P:.0f} mm forecast.\n"
           f"RED sinks: {len(reds)}  AMBER: {len(ambers)}.\n"
           + "\n".join(f"- sink {x.sink_id} fill {x.fill_ratio}" for x in reds.head(5).itertuples()))
    requests.post(f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage",
                  data={"chat_id": TELEGRAM_CHAT, "text": msg}, timeout=30)
    print("sent")

## Calibration plan (do this before telling anyone to act on an alert)
- The 0.5 / 1.0 fill-ratio thresholds are placeholders. Hindcast: run this notebook with
  historical rain (CHIRPS/IMD) for dates where notebook 03 observed real water, and tune the
  thresholds until POD/FAR are acceptable.
- This model ignores the storm-sewer network and pumping — alerts will over-predict near
  functioning drains. That bias is at least conservative for a warning system.
- Frame outputs as 'waterlogging likelihood', not certified flood forecasts.